In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
import random

In [0]:
spark = SparkSession.builder.appName("DataAnalysis_PySpark").getOrCreate()

In [0]:
schema = StructType([
    StructField("altitude", ArrayType(DoubleType()), True),
    StructField("gender", StringType(), True),
    StructField("heart_rate", ArrayType(LongType()), True),
    StructField("id", LongType(), True),
    StructField("latitude", ArrayType(DoubleType()), True),
    StructField("longitude", ArrayType(DoubleType()), True),
    StructField("speed", ArrayType(DoubleType()), True),
    StructField("sport", StringType(), True),
    StructField("timestamp", ArrayType(LongType()), True),
    StructField("url", StringType(), True),
    StructField("userId", LongType(), True)
])

In [0]:
def generate_data():
    sports = ["running", "cycling", "swimming", "hiking"]
    genders = ["male", "female", "other"]

    for x in range(201):
        yield(
           [random.uniform(0, 3000) for _ in range(5)],  # altitude
            random.choice(genders),                      # gender
            [random.randint(60, 180) for _ in range(5)], # heart_rate
            x,                                           # id
            [random.uniform(-90, 90) for _ in range(5)], # latitude
            [random.uniform(-180, 180) for _ in range(5)],# longitude
            [random.uniform(0, 50) for _ in range(5)],   # speed
            random.choice(sports),                      # sport
            [random.randint(1_600_000_000, 1_700_000_000) for _ in range(5)], # timestamp
            f"https://example.com/user_{x}",            # url
            random.randint(1, 1000)                     # userId 
        )

data = list(generate_data())
df = spark.createDataFrame(data, schema=schema)
df.show(5,truncate=True)


+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+--------+--------------------+--------------------+------+
|            altitude|gender|          heart_rate| id|            latitude|           longitude|               speed|   sport|           timestamp|                 url|userId|
+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+--------+--------------------+--------------------+------+
|[1409.50139867087...| other|[132, 176, 120, 6...|  0|[85.0988668335313...|[68.3692184959417...|[7.83960754945358...| cycling|[1650618890, 1620...|https://example.c...|   692|
|[2691.70504762513...|female|[70, 78, 62, 75, 75]|  1|[57.9766994103295...|[141.190050607534...|[37.6289194737778...|  hiking|[1657557334, 1657...|https://example.c...|   143|
|[1246.65273047983...| other|[159, 80, 178, 13...|  2|[78.6266984860265...|[61.4802919000782...|[2.45661889545872...| cy

In [0]:
df.createOrReplaceTempView('df_tbl')
spark.sql("SELECT * FROM df_tbl ").show(5)


+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+--------+--------------------+--------------------+------+
|            altitude|gender|          heart_rate| id|            latitude|           longitude|               speed|   sport|           timestamp|                 url|userId|
+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+--------+--------------------+--------------------+------+
|[1409.50139867087...| other|[132, 176, 120, 6...|  0|[85.0988668335313...|[68.3692184959417...|[7.83960754945358...| cycling|[1650618890, 1620...|https://example.c...|   692|
|[2691.70504762513...|female|[70, 78, 62, 75, 75]|  1|[57.9766994103295...|[141.190050607534...|[37.6289194737778...|  hiking|[1657557334, 1657...|https://example.c...|   143|
|[1246.65273047983...| other|[159, 80, 178, 13...|  2|[78.6266984860265...|[61.4802919000782...|[2.45661889545872...| cy